<a href="https://colab.research.google.com/github/YefridC09/ST-554-Project1-Template/blob/main/Task1/st554-project1-task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Title: ST-554 Project 1 Task 1 \
Author: Stephen Griggs \
Date: 2/22/2026

**As of 2/22/2026 the UCI ML repository's SSL certificate has expired and Google Colab can no longer download datasets through `ucimlrep`.**

Install the `ucimlrepo` package which is needed to download the UC Irvine Air Quality dataset.

In [1]:
#!pip install ucimlrepo

**Edit: Will be using `pandas` to read in .xlsx file from the UCI ML repository uploaded to Google Colab.**

Import the `fetch_ucirep` function for downloading data and `num_py` for math and matrix related operations.
Then, remove observations where the C6H6(GT) or CO(GT) are -200 which represent missing values.

In [2]:
#from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

#air_quality = fetch_ucirepo(id=360)
#air_quality = air_quality.data.features
air_quality = pd.read_excel("AirQualityUCI.xlsx")
air_quality = air_quality[
    (air_quality["C6H6(GT)"] != -200) &
    (air_quality["CO(GT)"] != -200)
]

## Grid Search with response variable $y$

**`calc_rmse(response, c)`** computes the root mean squared error between a column of observed values and a constant $c$, using the formula $\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - c)^2}$.

**`find_best_c(response, num_points=100)`** performs a grid search to find the optimal constant prediction:

1. **Compute quartiles:** Calculates the first quartile ($Q_1$) and third quartile ($Q_3$) of the response column to establish a reasonable search range.
2. **Build the grid:** Creates an evenly spaced grid of 100 candidate $c$ values between $Q_1$ and $Q_3$ using `np.linspace`.
3. **Evaluate RMSE:** Uses a list comprehension to loop over every $c$ in the grid, computing the RMSE for each candidate value.
4. **Find the optimum:** Pairs each $c$ with its corresponding RMSE, then selects the pair with the smallest RMSE using `min` with a lambda key.
5. **Report and return:** Prints the best $c$ and its associated RMSE, then returns the best $c$ as the optimal constant prediction.

In [3]:
# root mean square error function
def calc_rmse(response, c):
    return np.sqrt(1/len(response) * sum((response-c)**2))

def find_best_c(response, num_points=100):
    # q1 and q3 of response column
    q1, q3 = response.quantile([0.25, 0.75])
    grid = np.linspace(q1, q3, num_points)

    # loop over grid of c values finding rmse for each
    rmse_vals = [calc_rmse(response, c) for c in grid]
    paired = list(zip(grid, rmse_vals))

    # determine which value of c gives the smallest rmse
    best_c, min_rmse = min(paired, key=lambda x: x[1])

    # report best value of c as a prediction
    print(f"The best value of c is {best_c} with an RMSE of {min_rmse}")

    # return best value of c
    return best_c

- Run `find_best_c()` on `C6H6(GT)` and `PT08.S1(CO)` to find the optimal constant prediction for each variable.
- Compute the mean of each variable via `np.mean()` to verify that the optimal constant matches the sample mean.

In [4]:
print(find_best_c(air_quality["C6H6(GT)"]))
print(find_best_c(air_quality["PT08.S1(CO)"]))
print(np.mean(air_quality["C6H6(GT)"]))
print(np.mean(air_quality["PT08.S1(CO)"]))

The best value of c is 10.265037532914675 with an RMSE of 7.440576487435652
10.265037532914675
The best value of c is 1109.5852272727273 with an RMSE of 218.6720040572726
1109.5852272727273
10.275643255825724
1110.4551220951344


## Grid Search with response variable $y$ and numeric predictor $x$

1. **Build two grids:** Creates a grid of candidate $b_0$ values from $-25$ to $-15$ (step size $0.1$) and a grid of candidate $b_1$ values from $-5$ to $5$ (step size $0.01$) using `np.arange`.

2. **Evaluate RMSE over all pairs:** Uses a nested list comprehension to loop over every combination of $(b_0, b_1)$, computing $\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - (b_0 + b_1 x_i))^2}$ for each pair by passing the linear prediction $b_0 + b_1 \cdot x$ as the constant argument to `calc_rmse`.

3. **Find the optimum:** Selects the $(b_0, b_1, \text{RMSE})$ triple with the smallest RMSE using `min` with a lambda key on the third element.

4. **Report and return:** Prints the best $b_0$, $b_1$, and associated RMSE, then returns the optimal $(b_0, b_1)$ pair.

In [5]:
def find_best_c(response, predictor):
    # grid for predictor and response (beta0 and beta1)
    beta0_grid = np.arange(-25, -15, 0.1)
    beta1_grid = np.arange(-5, 5, 0.01)

    # loop over grid of c values finding rmse for each variable
    rmse_grid = [(b0, b1, calc_rmse(response, b0 + b1 * predictor))
           for b0 in beta0_grid
           for b1 in beta1_grid]

    # determine which vector c gives the smallest rmse
    best_b0, best_b1, min_rmse = min(rmse_grid, key=lambda x: x[2])

    # report best vector c as a prediction
    print(f"The best values for beta0 and beta1 are ({best_b0}, {best_b1} with an RMSE of {min_rmse}")

    # return best values of beta0 and beta1
    return best_b0, best_b1

Use the optimal $b_0$ and $b_1$ found by `find_best_c` to predict new `C6H6(GT)` values for `PT08.S1(CO)` inputs of $946$, $1075$, and $1246$ via the linear equation $\hat{y} = b_0 + b_1 \cdot x$.

In [6]:
# Find best coefficients using the data
beta0, beta1 = find_best_c(air_quality["C6H6(GT)"], air_quality["PT08.S1(CO)"])

# Predict C6H6(GT) for new PT08.S1(CO) values
new_values = np.array([946, 1075, 1246])
predictions = beta0 + beta1 * new_values

print(predictions)

The best values for beta0 and beta1 are (-22.99999999999997, 0.02999999999989278 with an RMSE of 3.542551139497846
[ 5.38  9.25 14.38]


## Gradient Descent with response variable $y$

The code implements a simple gradient descent algorithm to find the constant $c$ that minimizes the Root Mean Squared Error (RMSE) for a given response variable $y$.

1. **`calc_rmse(response, c)`** computes the root mean squared error between the observed values and a constant guess $c$:
$$RMSE(c) = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - c)^2}$$

2. **`diff_quotient(response, cur_c, delta)`** approximates the derivative (slope) of the RMSE function at the current guess using a numerical difference quotient:
$$\frac{RMSE(c + \Delta) - RMSE(c)}{\Delta}$$

3. **`gradient_descent(response, ...)`** iteratively searches for the optimal $c$:
   1. **Compute the slope**: Evaluates the difference quotient at the current guess `cur_c`.
   2. **Update the guess**: Moves `cur_c` a small step in the *negative* direction of the slope:
      $$c_{\text{new}} = c_{\text{cur}} - \text{step\_size} \times \text{diff\_quotient}$$
   3. **Check for convergence**: If `abs(new_c - cur_c) < tol`, the algorithm has converged and returns `new_c`.
   4. **Otherwise**, sets `cur_c = new_c` and repeats from step 1.
   5. **Safety stop**: If convergence isn't reached within `max_iter` iterations, it prints a warning and returns the last value of `cur_c`.

In [7]:
def calc_rmse(response, c):
    return np.sqrt(1/len(response) * sum((response-c)**2))

def diff_quotient(response, cur_c, delta):
    return (calc_rmse(response, cur_c + delta) - calc_rmse(response, cur_c)) / delta

def gradient_descent(response, cur_c=0, delta=0.001, step_size=0.01, tol=0.0001, max_iter=10000, debug=False):
    for i in range(max_iter):
        dq = diff_quotient(response, cur_c, delta)
        new_c = cur_c - dq * step_size
        if debug:
            print(f"Iter {i}: cur_c={cur_c}, dq={dq}, new_c={new_c}")
        if abs(new_c - cur_c) < tol:
            return new_c
        else:
          cur_c = new_c

    print(f"Warning: max iterations ({max_iter}) reached without convergence.")
    return cur_c

Run the gradient descent algorithm on the C6H6(GT) and PT08.S1(CO) columns respectively, each with tailored starting values and step sizes, to find the optimal constant prediction (the mean) for each variable.

In [8]:
print(gradient_descent(air_quality["C6H6(GT)"], cur_c=0, delta=0.001, step_size=0.01, tol=0.0001))
print(gradient_descent(air_quality["PT08.S1(CO)"], cur_c=1100, delta=0.001, step_size=0.1, tol=0.0001))

10.20090225404041
1110.2360710467474


## Gradient Descent with response variable $y$ and numeric predictor $x$

1. **`calc_rmse`** computes the Root Mean Squared Error: $RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y - b_0 - b_1 x)^2}$ for given values of $b_0$ and $b_1$.

2. **`diff_quotient_b0`** approximates the partial derivative of RMSE with respect to $b_0$ by computing the difference quotient: $\frac{RMSE(b_0 + \Delta_0,\; b_1) - RMSE(b_0,\; b_1)}{\Delta_0}$.

3. **`diff_quotient_b1`** approximates the partial derivative of RMSE with respect to $b_1$ by computing the difference quotient: $\frac{RMSE(b_0,\; b_1 + \Delta_1) - RMSE(b_0,\; b_1)}{\Delta_1}$.

4. **`gradient_descent2`** iteratively finds the optimal $b_0$ and $b_1$ that minimize RMSE:
   - **Initialize** starting guesses `cur_b0` and `cur_b1`.
   - **Each iteration:**
     1. Compute the difference quotient for $b_0$ at the current values and update: $b_0^{\text{new}} = b_0^{\text{cur}} - \text{step\_size}_{b_0} \cdot dq_0$.
     2. Compute the difference quotient for $b_1$ using the **newly updated** $b_0$ and the current $b_1$, then update: $b_1^{\text{new}} = b_1^{\text{cur}} - \text{step\_size}_{b_1} \cdot dq_1$.
     3. Compute the **Euclidean distance** between $(b_0^{\text{cur}}, b_1^{\text{cur}})$ and $(b_0^{\text{new}}, b_1^{\text{new}})$.
     4. If that distance is below the tolerance, **return** the new values as the final estimates.
     5. Otherwise, set the current values to the new values and repeat.
   - If the loop exhausts `max_iter` iterations without converging, it prints a warning and returns the last computed values.

In [9]:
def calc_rmse(response, predictor, b0, b1):
    return np.sqrt(np.mean((response - b0 - b1 * predictor) ** 2))

def diff_quotient_b0(response, predictor, b0, b1, delta0,):
    return (calc_rmse(response, predictor, b0 + delta0, b1) - calc_rmse(response, predictor, b0, b1)) / delta0

def diff_quotient_b1(response, predictor, b0, b1, delta1):
    return (calc_rmse(response, predictor, b0, b1 + delta1) - calc_rmse(response, predictor, b0, b1)) / delta1

def gradient_descent2(response, predictor, cur_b0=-20, cur_b1=0,
                     delta0=0.005, delta1=0.005,
                     step_size_b0=0.5, step_size_b1=0.00005,
                     tol=0.0001, max_iter=100000, debug=False):
    for i in range(max_iter):
        # update b0
        dq0 = diff_quotient_b0(response, predictor, cur_b0, cur_b1, delta0)
        new_b0 = cur_b0 - dq0 * step_size_b0

        # update b1 using new_b0
        dq1 = diff_quotient_b1(response, predictor, new_b0, cur_b1, delta1)
        new_b1 = cur_b1 - dq1 * step_size_b1

        if debug:
            print(f"Iter {i}: b0={new_b0}, b1={new_b1}, dq={dq0}, dq1={dq1}")

        # check Euclidean distance
        dist = np.linalg.norm(np.array([new_b0, new_b1]) - np.array([cur_b0, cur_b1]))

        if dist < tol:
            return new_b0, new_b1
        else:
          cur_b0, cur_b1 = new_b0, new_b1

    print(f"Warning: max iterations ({max_iter}) reached without convergence.")
    return cur_b0, cur_b1

Run the gradient descent algorithm on the air quality dataset to find the optimal intercept ($b_0$) and slope ($b_1$) that minimize the RMSE for predicting `C6H6(GT)` from `PT08.S1(CO)`, storing the results in `beta0` and `beta1`.

**Note: this code didn't converge. It timed out after 2 minutes at a maximum of 100000 iterations.**

In [10]:
beta0, beta1 = gradient_descent2(air_quality["C6H6(GT)"], air_quality["PT08.S1(CO)"],
                  cur_b0=-20, cur_b1=0,
                  delta0=0.005, delta1=0.005,
                  step_size_b0=0.5, step_size_b1=0.00005,
                  tol=0.0001, max_iter=100000, debug=False)

In [11]:
print((beta0, beta1))

(np.float64(-22.301583378427097), np.float64(-0.0014869831391278593))


Use the optimized $b_0$ and $b_1$ values to predict `C6H6(GT)` for three new `PT08.S1(CO)` values (946, 1075, and 1246) using the linear equation $\hat{y} = b_0 + b_1 x$.

In [12]:
# Predict C6H6(GT) for new PT08.S1(CO) values
new_values = np.array([946, 1075, 1246])
predictions = beta0 + beta1 * new_values

print(predictions)

[-23.70826943 -23.90009025 -24.15436437]
